In [1]:
import platform
import sys
import json
import zipfile
import getpass

import pandas as pd
import mysql.connector

print("PharmaOpt Notebook 05")
print("----------------------")
print("Operating system:", platform.system())
print("Python location:", sys.executable)
print("Pandas version:", pd.__version__)
print("Setup successful.")

PharmaOpt Notebook 05
----------------------
Operating system: Windows
Python location: C:\Users\jpiza\anaconda3\envs\pharmaopt\python.exe
Pandas version: 3.0.6
Setup successful.


In [2]:
mysql_password = getpass.getpass("Enter your MySQL password: ")

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=mysql_password,
    database="pharmaopt"
)

print("Connected to PharmaOpt:", connection.is_connected())

Enter your MySQL password:  ········


Connected to PharmaOpt: True


In [3]:
cursor = connection.cursor()

cursor.execute("SHOW TABLES;")
tables = cursor.fetchall()

print("Tables in PharmaOpt:")
for table in tables:
    print("-", table[0])

Tables in PharmaOpt:
- drug_package
- drug_product
- recall_event
- recall_package
- shortage_event


In [4]:
def load_fda_zip(zip_filename):
    with zipfile.ZipFile(zip_filename, "r") as z:
        json_filename = z.namelist()[0]

        with z.open(json_filename) as f:
            data = json.load(f)

    return pd.json_normalize(data["results"])

print("FDA ZIP loader ready.")

FDA ZIP loader ready.


In [5]:
shortages = load_fda_zip(
    "drug-shortages-0001-of-0001.json.zip"
)

ndc = load_fda_zip(
    "drug-ndc-0001-of-0001.json.zip"
)

recalls = load_fda_zip(
    "drug-enforcement-0001-of-0001.json.zip"
)

print("FDA datasets loaded successfully.")
print("-------------------------------")
print("Shortage records:", len(shortages))
print("NDC product records:", len(ndc))
print("Recall records:", len(recalls))

FDA datasets loaded successfully.
-------------------------------
Shortage records: 1601
NDC product records: 138217
Recall records: 17975


In [6]:
package_records = []

for _, row in ndc.iterrows():
    product_ndc = row.get("product_ndc")
    packaging = row.get("packaging")

    if isinstance(packaging, list):
        for package in packaging:
            package_records.append({
                "product_ndc": product_ndc,
                "package_ndc": package.get("package_ndc"),
                "package_description": package.get("description")
            })

packages = pd.DataFrame(package_records)

packages["product_ndc"] = packages["product_ndc"].astype("string").str.strip()
packages["package_ndc"] = packages["package_ndc"].astype("string").str.strip()

print("Package rows:", len(packages))
print("Unique package NDCs:", packages["package_ndc"].nunique())

packages.head()

Package rows: 256387
Unique package NDCs: 256383


,product_ndc,package_ndc,package_description
0,71288-432,71288-432-81,10 SYRINGE in 1 CARTON (71288-432-81) / .3 mL...
1,71288-440,71288-440-02,10 VIAL in 1 CARTON (71288-440-02) / 1 mL in ...
2,71288-441,71288-441-10,1 VIAL in 1 CARTON (71288-441-10) / 10 mL in ...
3,0409-0183,0409-0183-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0409-0183-01)..."
4,71288-450,71288-450-02,"25 VIAL, MULTI-DOSE in 1 CARTON (71288-450-02)..."


In [7]:
recall_packages = recalls[
    [
        "event_id",
        "recall_number",
        "recalling_firm",
        "classification",
        "reason_for_recall",
        "recall_initiation_date",
        "openfda.package_ndc"
    ]
].copy()

recall_packages = recall_packages.explode("openfda.package_ndc")

recall_packages = recall_packages.rename(
    columns={"openfda.package_ndc": "package_ndc"}
)

recall_packages["package_ndc"] = (
    recall_packages["package_ndc"]
    .astype("string")
    .str.strip()
)

print("Recall-package rows:", len(recall_packages))
print(
    "Unique recall package NDCs:",
    recall_packages["package_ndc"].nunique()
)

recall_packages.head()

Recall-package rows: 40316
Unique recall package NDCs: 11233


,event_id,recall_number,recalling_firm,classification,reason_for_recall,recall_initiation_date,package_ndc
0,99666,N/A,"Safecor Health, LLC",Not Yet Classified,Cross Contamination with Other Products,20260814,<NA>
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1721-93
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1722-93
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1723-93
1,81707,D-0321-2019,"Mylan Laboratories Limited, (Nashik FDF)",Class II,GCMP Deviations: FDA analysis confirmed presen...,20181204,0378-1724-93


In [8]:
package_ndc_set = set(
    packages["package_ndc"].dropna().unique()
)

shortages["source_package_ndc"] = (
    shortages["package_ndc"]
    .astype("string")
    .str.strip()
)

shortages["linked_package_ndc"] = (
    shortages["source_package_ndc"]
    .where(shortages["source_package_ndc"].isin(package_ndc_set))
)

unique_shortage_ndcs = shortages["source_package_ndc"].nunique()

matched_shortage_ndcs = shortages.loc[
    shortages["linked_package_ndc"].notna(),
    "source_package_ndc"
].nunique()

match_rate = (
    matched_shortage_ndcs / unique_shortage_ndcs * 100
)

print("SHORTAGE → NDC INTEGRATION")
print("--------------------------------")
print("Unique shortage NDCs:", unique_shortage_ndcs)
print("Matched NDCs:", matched_shortage_ndcs)
print(f"Match rate: {match_rate:.2f}%")

SHORTAGE → NDC INTEGRATION
--------------------------------
Unique shortage NDCs: 1570
Matched NDCs: 1393
Match rate: 88.73%


In [9]:
print("PHARMAOPT DATABASE LOAD VALIDATION")
print("=" * 50)

print("\nDRUG PRODUCT")
print("Rows:", len(ndc))
print("Unique product_ndc:", ndc["product_ndc"].nunique())
print("Missing product_ndc:", ndc["product_ndc"].isna().sum())
print(
    "Duplicate product_ndc:",
    ndc["product_ndc"].duplicated().sum()
)

print("\nDRUG PACKAGE")
print("Rows:", len(packages))
print(
    "Unique package_ndc:",
    packages["package_ndc"].nunique()
)
print(
    "Missing package_ndc:",
    packages["package_ndc"].isna().sum()
)
print(
    "Duplicate package_ndc:",
    packages["package_ndc"].duplicated().sum()
)

print("\nRECALL EVENT")
print("Rows:", len(recalls))
print(
    "Unique recall_number:",
    recalls["recall_number"].nunique()
)
print(
    "Missing recall_number:",
    recalls["recall_number"].isna().sum()
)
print(
    "Duplicate recall_number:",
    recalls["recall_number"].duplicated().sum()
)

print("\nSHORTAGE EVENT")
print("Rows:", len(shortages))
print(
    "Unique source package NDCs:",
    shortages["source_package_ndc"].nunique()
)
print(
    "Linked rows:",
    shortages["linked_package_ndc"].notna().sum()
)
print(
    "Unlinked rows:",
    shortages["linked_package_ndc"].isna().sum()
)

PHARMAOPT DATABASE LOAD VALIDATION

DRUG PRODUCT
Rows: 138217
Unique product_ndc: 135772
Missing product_ndc: 0
Duplicate product_ndc: 2445

DRUG PACKAGE
Rows: 256387
Unique package_ndc: 256383
Missing package_ndc: 0
Duplicate package_ndc: 4

RECALL EVENT
Rows: 17975
Unique recall_number: 17975
Missing recall_number: 0
Duplicate recall_number: 0

SHORTAGE EVENT
Rows: 1601
Unique source package NDCs: 1570
Linked rows: 1421
Unlinked rows: 180


In [10]:
product_columns = [
    "product_ndc",
    "generic_name",
    "brand_name",
    "labeler_name",
    "dosage_form",
    "marketing_category"
]

product_duplicates = (
    ndc.loc[
        ndc["product_ndc"].duplicated(keep=False),
        product_columns
    ]
    .sort_values("product_ndc")
)

product_duplicate_summary = (
    product_duplicates
    .groupby("product_ndc")
    .nunique(dropna=False)
)

product_conflicts = product_duplicate_summary[
    (product_duplicate_summary > 1).any(axis=1)
]

print("PRODUCT DUPLICATE AUDIT")
print("=" * 45)
print(
    "Product NDCs appearing more than once:",
    product_duplicates["product_ndc"].nunique()
)
print(
    "Product NDCs with conflicting database fields:",
    len(product_conflicts)
)

product_duplicates.head(20)

PRODUCT DUPLICATE AUDIT
Product NDCs appearing more than once: 1910
Product NDCs with conflicting database fields: 954


,product_ndc,generic_name,brand_name,labeler_name,dosage_form,marketing_category
34475,0002-0800,diluent,Sterile Diluent,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
71441,0002-0800,diluent,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
72918,0002-1436,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
71442,0002-1436,galcanezumab-gnlm,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
112540,0002-2377,galcanezumab-gnlm,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
47778,0002-2377,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
112543,0002-3115,galcanezumab-gnlm,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
117541,0002-3115,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
62150,0009-0039,methylprednisolone sodium succinate,SOLU-MEDROL,Pharmacia & Upjohn Company LLC,"INJECTION, POWDER, FOR SOLUTION",NDA
34749,0009-0039,methylprednisolone sodium succinate,SOLU-MEDROL,Pharmacia & Upjohn Company LLC,"INJECTION, POWDER, FOR SOLUTION",NDA


In [11]:
package_duplicates = (
    packages.loc[
        packages["package_ndc"].duplicated(keep=False)
    ]
    .sort_values("package_ndc")
)

package_duplicate_summary = (
    package_duplicates
    .groupby("package_ndc")
    [["product_ndc", "package_description"]]
    .nunique(dropna=False)
)

package_conflicts = package_duplicate_summary[
    (package_duplicate_summary > 1).any(axis=1)
]

print("PACKAGE DUPLICATE AUDIT")
print("=" * 45)
print(
    "Package NDCs appearing more than once:",
    package_duplicates["package_ndc"].nunique()
)
print(
    "Package NDCs with conflicting information:",
    len(package_conflicts)
)

package_duplicates

PACKAGE DUPLICATE AUDIT
Package NDCs appearing more than once: 4
Package NDCs with conflicting information: 0


,product_ndc,package_ndc,package_description
254948,72043-2500,72043-2500-1,48 g in 1 BOTTLE (72043-2500-1)
254949,72043-2500,72043-2500-1,48 g in 1 BOTTLE (72043-2500-1)
128499,72603-133,72603-133-25,25 VIAL in 1 CARTON (72603-133-25) / 2 mL in ...
128500,72603-133,72603-133-25,25 VIAL in 1 CARTON (72603-133-25) / 2 mL in ...
128512,72603-251,72603-251-25,25 VIAL in 1 CARTON (72603-251-25) / 4 mL in ...
128513,72603-251,72603-251-25,25 VIAL in 1 CARTON (72603-251-25) / 4 mL in ...
27004,72603-474,72603-474-25,25 VIAL in 1 CARTON (72603-474-25) / 10 mL in...
27005,72603-474,72603-474-25,25 VIAL in 1 CARTON (72603-474-25) / 10 mL in...


In [12]:
valid_recall_packages = recall_packages[
    recall_packages["package_ndc"].notna()
    & (recall_packages["package_ndc"] != "")
    & (recall_packages["package_ndc"] != "<NA>")
].copy()

print("RECALL PACKAGE VALIDATION")
print("=" * 45)
print("Exploded rows:", len(recall_packages))
print("Rows with a package NDC:", len(valid_recall_packages))
print(
    "Rows without a package NDC:",
    len(recall_packages) - len(valid_recall_packages)
)
print(
    "Unique package NDCs:",
    valid_recall_packages["package_ndc"].nunique()
)

RECALL PACKAGE VALIDATION
Exploded rows: 40316
Rows with a package NDC: 25577
Rows without a package NDC: 14739
Unique package NDCs: 11233


In [13]:
# Keep FDA's original recall package NDC
valid_recall_packages["source_package_ndc"] = (
    valid_recall_packages["package_ndc"]
    .astype("string")
    .str.strip()
)

# Link only when that package exists in our current NDC directory
valid_recall_packages["linked_package_ndc"] = (
    valid_recall_packages["source_package_ndc"]
    .where(
        valid_recall_packages["source_package_ndc"].isin(package_ndc_set)
    )
)

print("RECALL → CURRENT NDC MATCHING")
print("=" * 45)

print("Recall-package rows:", len(valid_recall_packages))

print(
    "Linked rows:",
    valid_recall_packages["linked_package_ndc"].notna().sum()
)

print(
    "Unlinked rows:",
    valid_recall_packages["linked_package_ndc"].isna().sum()
)

print(
    "Unique source package NDCs:",
    valid_recall_packages["source_package_ndc"].nunique()
)

print(
    "Unique linked package NDCs:",
    valid_recall_packages["linked_package_ndc"].nunique()
)

RECALL → CURRENT NDC MATCHING
Recall-package rows: 25577
Linked rows: 19673
Unlinked rows: 5904
Unique source package NDCs: 11233
Unique linked package NDCs: 8606


In [14]:
product_fields = [
    "generic_name",
    "brand_name",
    "labeler_name",
    "dosage_form",
    "marketing_category"
]

duplicate_products = ndc[
    ndc["product_ndc"].duplicated(keep=False)
].copy()

# Count DISTINCT NON-MISSING values for each field
true_value_counts = (
    duplicate_products
    .groupby("product_ndc")[product_fields]
    .nunique(dropna=True)
)

# A real conflict means the same product_ndc has
# more than one actual non-null value for a field
true_conflicts = true_value_counts[
    (true_value_counts > 1).any(axis=1)
]

print("REFINED PRODUCT DUPLICATE AUDIT")
print("=" * 50)

print(
    "Product NDCs appearing more than once:",
    duplicate_products["product_ndc"].nunique()
)

print(
    "Product NDCs with >1 real non-null value:",
    len(true_conflicts)
)

print(
    "Duplicates explainable by identical/missing values:",
    duplicate_products["product_ndc"].nunique()
    - len(true_conflicts)
)

print("\nREAL CONFLICTS BY FIELD")
print("-" * 50)

for column in product_fields:
    count = (true_value_counts[column] > 1).sum()
    print(f"{column}: {count}")

REFINED PRODUCT DUPLICATE AUDIT
Product NDCs appearing more than once: 1910
Product NDCs with >1 real non-null value: 954
Duplicates explainable by identical/missing values: 956

REAL CONFLICTS BY FIELD
--------------------------------------------------
generic_name: 156
brand_name: 38
labeler_name: 144
dosage_form: 4
marketing_category: 779


In [15]:
conflicting_ndcs = true_conflicts.index.tolist()

conflict_examples = (
    duplicate_products[
        duplicate_products["product_ndc"].isin(conflicting_ndcs)
    ][
        [
            "product_ndc",
            "generic_name",
            "brand_name",
            "labeler_name",
            "dosage_form",
            "marketing_category"
        ]
    ]
    .sort_values("product_ndc")
)

print("Number of rows involved:", len(conflict_examples))

conflict_examples.head(30)

Number of rows involved: 2112


,product_ndc,generic_name,brand_name,labeler_name,dosage_form,marketing_category
71441,0002-0800,diluent,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
34475,0002-0800,diluent,Sterile Diluent,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
71442,0002-1436,galcanezumab-gnlm,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
72918,0002-1436,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
47778,0002-2377,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
112540,0002-2377,galcanezumab-gnlm,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
112543,0002-3115,galcanezumab-gnlm,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING
117541,0002-3115,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,"INJECTION, SOLUTION",BLA
79630,0049-3190,Voriconazole,VFEND,Roerig,"INJECTION, POWDER, LYOPHILIZED, FOR SOLUTION",NDA
107012,0049-3190,voriconazole,VFEND,Roerig,"INJECTION, POWDER, LYOPHILIZED, FOR SOLUTION",NDA


In [16]:
print("FDA PRODUCT_ID VALIDATION")
print("=" * 50)

print("product_id column exists:", "product_id" in ndc.columns)

if "product_id" in ndc.columns:
    print("Total rows:", len(ndc))
    print("Unique product_id:", ndc["product_id"].nunique())
    print("Missing product_id:", ndc["product_id"].isna().sum())
    print("Duplicate product_id:", ndc["product_id"].duplicated().sum())

    print("\nExample records:")
    
    display(
        ndc.loc[
            ndc["product_ndc"].isin(
                ["0002-0800", "0002-1436", "0002-2377"]
            ),
            [
                "product_id",
                "product_ndc",
                "spl_id",
                "generic_name",
                "brand_name",
                "labeler_name",
                "marketing_category"
            ]
        ].sort_values("product_ndc")
    )

FDA PRODUCT_ID VALIDATION
product_id column exists: True
Total rows: 138217
Unique product_id: 138217
Missing product_id: 0
Duplicate product_id: 0

Example records:


,product_id,product_ndc,spl_id,generic_name,brand_name,labeler_name,marketing_category
34475,0002-0800_03be5386-187a-420f-9021-55778a73eba0,0002-0800,03be5386-187a-420f-9021-55778a73eba0,diluent,Sterile Diluent,Eli Lilly and Company,BLA
71441,0002-0800_55aa4078-e1e8-4338-b2b5-a9298b3c16e2,0002-0800,55aa4078-e1e8-4338-b2b5-a9298b3c16e2,diluent,NaN,Eli Lilly and Company,DRUG FOR FURTHER PROCESSING
71442,0002-1436_368df5d9-5eeb-49c7-a0dd-8a326ee5d78f,0002-1436,368df5d9-5eeb-49c7-a0dd-8a326ee5d78f,galcanezumab-gnlm,NaN,Eli Lilly and Company,DRUG FOR FURTHER PROCESSING
72918,0002-1436_b58a1b24-8a42-4073-9413-4f9b7c308a64,0002-1436,b58a1b24-8a42-4073-9413-4f9b7c308a64,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,BLA
47778,0002-2377_b58a1b24-8a42-4073-9413-4f9b7c308a64,0002-2377,b58a1b24-8a42-4073-9413-4f9b7c308a64,galcanezumab-gnlm,EMGALITY,Eli Lilly and Company,BLA
112540,0002-2377_368df5d9-5eeb-49c7-a0dd-8a326ee5d78f,0002-2377,368df5d9-5eeb-49c7-a0dd-8a326ee5d78f,galcanezumab-gnlm,NaN,Eli Lilly and Company,DRUG FOR FURTHER PROCESSING


In [17]:
print("PRODUCT_ID FINAL VALIDATION")
print("=" * 50)

print("Total NDC rows:", len(ndc))
print("Unique product_id:", ndc["product_id"].nunique())
print("Missing product_id:", ndc["product_id"].isna().sum())
print("Duplicate product_id:", ndc["product_id"].duplicated().sum())

PRODUCT_ID FINAL VALIDATION
Total NDC rows: 138217
Unique product_id: 138217
Missing product_id: 0
Duplicate product_id: 0


In [18]:
package_records_v2 = []

for _, row in ndc.iterrows():

    product_id = row.get("product_id")
    product_ndc = row.get("product_ndc")
    packaging = row.get("packaging")

    if isinstance(packaging, list):

        for package in packaging:

            package_records_v2.append({
                "product_id": product_id,
                "product_ndc": product_ndc,
                "package_ndc": package.get("package_ndc"),
                "package_description": package.get("description")
            })

packages_v2 = pd.DataFrame(package_records_v2)

for column in ["product_id", "product_ndc", "package_ndc"]:
    packages_v2[column] = (
        packages_v2[column]
        .astype("string")
        .str.strip()
    )

print("PACKAGE → PRODUCT_ID VALIDATION")
print("=" * 50)

print("Package relationship rows:", len(packages_v2))
print("Unique package NDCs:", packages_v2["package_ndc"].nunique())

package_product_counts = (
    packages_v2
    .groupby("package_ndc")["product_id"]
    .nunique()
)

print(
    "Package NDCs linked to multiple product_ids:",
    (package_product_counts > 1).sum()
)

display(
    packages_v2[
        packages_v2["package_ndc"].isin(
            package_product_counts[
                package_product_counts > 1
            ].index
        )
    ].sort_values("package_ndc").head(30)
)

PACKAGE → PRODUCT_ID VALIDATION
Package relationship rows: 256387
Unique package NDCs: 256383
Package NDCs linked to multiple product_ids: 0


,product_id,product_ndc,package_ndc,package_description


In [19]:
print("PRODUCT_ID FINAL VALIDATION")
print("=" * 50)

print("Total NDC rows:", len(ndc))
print("Unique product_id:", ndc["product_id"].nunique())
print("Missing product_id:", ndc["product_id"].isna().sum())
print("Duplicate product_id:", ndc["product_id"].duplicated().sum())

PRODUCT_ID FINAL VALIDATION
Total NDC rows: 138217
Unique product_id: 138217
Missing product_id: 0
Duplicate product_id: 0


In [20]:
def max_text_length(series):
    values = series.dropna().astype(str)
    return values.str.len().max() if len(values) else 0

print("DRUG PRODUCT FIELD SIZE AUDIT")
print("=" * 55)

fields = [
    "product_id",
    "product_ndc",
    "spl_id",
    "generic_name",
    "brand_name",
    "labeler_name",
    "dosage_form",
    "marketing_category"
]

for field in fields:
    print(
        f"{field:<25}",
        "max length:",
        max_text_length(ndc[field])
    )

print("\nDRUG PACKAGE FIELD SIZE AUDIT")
print("=" * 55)

for field in [
    "product_id",
    "product_ndc",
    "package_ndc",
    "package_description"
]:
    print(
        f"{field:<25}",
        "max length:",
        max_text_length(packages_v2[field])
    )

DRUG PRODUCT FIELD SIZE AUDIT
product_id                max length: 47
product_ndc               max length: 10
spl_id                    max length: 36
generic_name              max length: 512
brand_name                max length: 367
labeler_name              max length: 145
dosage_form               max length: 64
marketing_category        max length: 50

DRUG PACKAGE FIELD SIZE AUDIT
product_id                max length: 47
product_ndc               max length: 10
package_ndc               max length: 12
package_description       max length: 2429


In [22]:
# ============================================================
# REFRESH MYSQL CONNECTION AFTER SCHEMA REBUILD
# ============================================================

# Close the old cursor/connection if they are still open
try:
    cursor.close()
except:
    pass

try:
    connection.close()
except:
    pass

print("Old MySQL connection closed.")

# Ask for password again
mysql_password = getpass.getpass("Enter your MySQL password: ")

# Create a fresh connection to the rebuilt PharmaOpt database
connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=mysql_password,
    database="pharmaopt"
)

cursor = connection.cursor(buffered=True)

print("Connected to PharmaOpt:", connection.is_connected())

# Get the CURRENT tables
cursor.execute("SHOW TABLES;")
tables = [row[0] for row in cursor.fetchall()]

print("\nPHARMAOPT TABLE CHECK")
print("=" * 50)

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`;")
    count = cursor.fetchone()[0]
    print(f"{table:<20} {count:,} rows")

Old MySQL connection closed.


Enter your MySQL password:  ········


Connected to PharmaOpt: True

PHARMAOPT TABLE CHECK
drug_package         0 rows
drug_product         0 rows
recall_event         0 rows
recall_package       0 rows
shortage_event       0 rows


In [23]:
# ============================================================
# BUILD FINAL DRUG PRODUCT DATAFRAME
# ============================================================

drug_products_df = ndc[
    [
        "product_id",
        "product_ndc",
        "spl_id",
        "generic_name",
        "brand_name",
        "labeler_name",
        "dosage_form",
        "marketing_category"
    ]
].copy()

# Clean identifier fields
for column in ["product_id", "product_ndc", "spl_id"]:
    drug_products_df[column] = (
        drug_products_df[column]
        .astype("string")
        .str.strip()
    )

print("DRUG PRODUCT READY")
print("=" * 50)
print("Rows:", len(drug_products_df))
print("Unique product_id:", drug_products_df["product_id"].nunique())
print("Duplicate product_id:", drug_products_df["product_id"].duplicated().sum())
print("Missing product_id:", drug_products_df["product_id"].isna().sum())

DRUG PRODUCT READY
Rows: 138217
Unique product_id: 138217
Duplicate product_id: 0
Missing product_id: 0


In [24]:
# ============================================================
# BUILD FINAL DRUG PACKAGE DATAFRAME
# ============================================================

drug_packages_df = packages_v2[
    [
        "package_ndc",
        "product_id",
        "package_description"
    ]
].copy()

# We already audited these duplicates:
# 4 duplicate package NDCs, 0 conflicting records.
drug_packages_df = (
    drug_packages_df
    .drop_duplicates(subset=["package_ndc"], keep="first")
    .reset_index(drop=True)
)

valid_product_ids = set(drug_products_df["product_id"].dropna())

invalid_product_links = (
    ~drug_packages_df["product_id"].isin(valid_product_ids)
).sum()

print("DRUG PACKAGE READY")
print("=" * 50)
print("Rows:", len(drug_packages_df))
print("Unique package_ndc:", drug_packages_df["package_ndc"].nunique())
print("Duplicate package_ndc:", drug_packages_df["package_ndc"].duplicated().sum())
print("Missing package_ndc:", drug_packages_df["package_ndc"].isna().sum())
print("Invalid product_id foreign keys:", invalid_product_links)

DRUG PACKAGE READY
Rows: 256383
Unique package_ndc: 256383
Duplicate package_ndc: 0
Missing package_ndc: 0
Invalid product_id foreign keys: 0


In [25]:
# ============================================================
# MYSQL VALUE CLEANER
# ============================================================

def mysql_value(value):
    if pd.isna(value):
        return None
    
    # Convert pandas/numpy scalar values to ordinary Python values
    if hasattr(value, "item"):
        try:
            return value.item()
        except:
            pass
    
    return value

print("MySQL value cleaner ready.")

MySQL value cleaner ready.


In [26]:
# ============================================================
# LOAD DRUG PRODUCT → MYSQL
# ============================================================

product_insert_sql = """
INSERT INTO drug_product (
    product_id,
    product_ndc,
    spl_id,
    generic_name,
    brand_name,
    labeler_name,
    dosage_form,
    marketing_category
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

product_rows = [
    tuple(mysql_value(value) for value in row)
    for row in drug_products_df.itertuples(index=False, name=None)
]

batch_size = 5000

try:
    print("Loading drug_product...")

    for start in range(0, len(product_rows), batch_size):
        batch = product_rows[start:start + batch_size]

        cursor.executemany(
            product_insert_sql,
            batch
        )

        end = min(start + batch_size, len(product_rows))

        print(
            f"\rLoaded {end:,} / {len(product_rows):,}",
            end=""
        )

    connection.commit()

    print("\n\nDRUG PRODUCT LOAD COMPLETE.")

except Exception as e:
    connection.rollback()

    print("\nLOAD FAILED — transaction rolled back.")
    print(type(e).__name__ + ":", e)

Loading drug_product...
Loaded 138,217 / 138,217

DRUG PRODUCT LOAD COMPLETE.


In [27]:
cursor.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS unique_product_ids,
    COUNT(DISTINCT product_ndc) AS unique_product_ndcs
FROM drug_product;
""")

result = cursor.fetchone()

print("MYSQL DRUG PRODUCT VALIDATION")
print("=" * 50)
print("Total rows:", f"{result[0]:,}")
print("Unique product IDs:", f"{result[1]:,}")
print("Unique product NDCs:", f"{result[2]:,}")

MYSQL DRUG PRODUCT VALIDATION
Total rows: 138,217
Unique product IDs: 138,217
Unique product NDCs: 135,772


In [28]:
# ============================================================
# LOAD DRUG PACKAGE → MYSQL
# ============================================================

package_insert_sql = """
INSERT INTO drug_package (
    package_ndc,
    product_id,
    package_description
)
VALUES (%s, %s, %s)
"""

package_rows = [
    tuple(mysql_value(value) for value in row)
    for row in drug_packages_df.itertuples(index=False, name=None)
]

try:
    print("Loading drug_package...")

    for start in range(0, len(package_rows), batch_size):
        batch = package_rows[start:start + batch_size]

        cursor.executemany(
            package_insert_sql,
            batch
        )

        end = min(start + batch_size, len(package_rows))

        print(
            f"\rLoaded {end:,} / {len(package_rows):,}",
            end=""
        )

    connection.commit()

    print("\n\nDRUG PACKAGE LOAD COMPLETE.")

except Exception as e:
    connection.rollback()

    print("\nLOAD FAILED — transaction rolled back.")
    print(type(e).__name__ + ":", e)

Loading drug_package...
Loaded 256,383 / 256,383

DRUG PACKAGE LOAD COMPLETE.


In [30]:
cursor.execute("""
SELECT
    COUNT(*) AS package_rows,
    COUNT(DISTINCT package_ndc) AS unique_packages
FROM drug_package;
""")

package_result = cursor.fetchone()

cursor.execute("""
SELECT COUNT(*)
FROM drug_package p
LEFT JOIN drug_product d
    ON p.product_id = d.product_id
WHERE d.product_id IS NULL;
""")

orphan_packages = cursor.fetchone()[0]

print("MYSQL PACKAGE VALIDATION")
print("=" * 50)
print("Package rows:", f"{package_result[0]:,}")
print("Unique packages:", f"{package_result[1]:,}")
print("Packages without a valid product:", f"{orphan_packages:,}")

MYSQL PACKAGE VALIDATION
Package rows: 256,383
Unique packages: 256,383
Packages without a valid product: 0


In [31]:
print("SHORTAGE COLUMNS")
print("=" * 50)

for column in shortages.columns:
    print(column)

SHORTAGE COLUMNS
update_type
initial_posting_date
package_ndc
generic_name
contact_info
availability
update_date
therapeutic_category
dosage_form
presentation
company_name
status
openfda.application_number
openfda.brand_name
openfda.generic_name
openfda.manufacturer_name
openfda.product_ndc
openfda.product_type
openfda.route
openfda.substance_name
openfda.rxcui
openfda.spl_id
openfda.spl_set_id
openfda.package_ndc
openfda.unii
related_info
shortage_reason
openfda.nui
openfda.pharm_class_pe
openfda.pharm_class_epc
discontinued_date
related_info_link
openfda.pharm_class_cs
change_date
openfda.pharm_class_moa
resolved_note
source_package_ndc
linked_package_ndc


In [32]:
# ============================================================
# BUILD FINAL SHORTAGE EVENT DATAFRAME
# ============================================================

def safe_text(value):
    """Convert lists/dicts to JSON text while preserving nulls."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)

    return str(value).strip()


def safe_date_series(series):
    """Convert mixed FDA date strings to Python dates."""
    converted = pd.to_datetime(
        series,
        errors="coerce",
        format="mixed"
    )

    return converted.dt.date


shortages_df = shortages[
    [
        "source_package_ndc",
        "linked_package_ndc",
        "generic_name",
        "company_name",
        "status",
        "availability",
        "shortage_reason",
        "therapeutic_category",
        "dosage_form",
        "presentation",
        "update_type",
        "initial_posting_date",
        "update_date",
        "change_date",
        "resolved_note"
    ]
].copy()


# Text fields that may contain complex/list values
text_columns = [
    "generic_name",
    "company_name",
    "status",
    "availability",
    "shortage_reason",
    "therapeutic_category",
    "dosage_form",
    "presentation",
    "update_type",
    "resolved_note"
]

for column in text_columns:
    shortages_df[column] = shortages_df[column].apply(safe_text)


# Convert dates
date_columns = [
    "initial_posting_date",
    "update_date",
    "change_date"
]

for column in date_columns:
    shortages_df[column] = safe_date_series(
        shortages_df[column]
    )


print("SHORTAGE DATAFRAME BUILT")
print("=" * 50)
print("Rows:", len(shortages_df))
print(
    "Missing source NDC:",
    shortages_df["source_package_ndc"].isna().sum()
)
print(
    "Linked rows:",
    shortages_df["linked_package_ndc"].notna().sum()
)
print(
    "Unlinked rows:",
    shortages_df["linked_package_ndc"].isna().sum()
)

SHORTAGE DATAFRAME BUILT
Rows: 1601
Missing source NDC: 0
Linked rows: 1421
Unlinked rows: 180


In [33]:
# ============================================================
# SHORTAGE PRE-LOAD VALIDATION
# ============================================================

print("SHORTAGE PRE-LOAD VALIDATION")
print("=" * 55)

for column in date_columns:
    print(
        f"{column:<25}",
        "valid:",
        shortages_df[column].notna().sum(),
        "| missing/unparsed:",
        shortages_df[column].isna().sum()
    )


print("\nMAXIMUM TEXT LENGTHS")
print("-" * 55)

for column in text_columns:
    values = shortages_df[column].dropna().astype(str)

    max_length = (
        values.str.len().max()
        if len(values) > 0
        else 0
    )

    print(f"{column:<25} {max_length}")

SHORTAGE PRE-LOAD VALIDATION
initial_posting_date      valid: 1601 | missing/unparsed: 0
update_date               valid: 1601 | missing/unparsed: 0
change_date               valid: 11 | missing/unparsed: 1590

MAXIMUM TEXT LENGTHS
-------------------------------------------------------
generic_name              139
company_name              87
status                    18
availability              90
shortage_reason           67
therapeutic_category      165
dosage_form               25
presentation              325
update_type               10
resolved_note             30


In [35]:
# ============================================================
# VERIFY SHORTAGE FOREIGN KEYS
# ============================================================

# Build the package set ONCE
valid_package_ndcs = set(
    drug_packages_df["package_ndc"]
    .dropna()
    .astype(str)
)

# Get unique linked shortage NDCs
linked_shortage_ndcs = set(
    shortages_df["linked_package_ndc"]
    .dropna()
    .astype(str)
)

# Compare the two sets directly
invalid_links = linked_shortage_ndcs - valid_package_ndcs

print("SHORTAGE FOREIGN KEY CHECK")
print("=" * 50)
print("Unique linked NDCs:", len(linked_shortage_ndcs))
print("Invalid links:", len(invalid_links))

if invalid_links:
    print("\nExample invalid links:")
    print(list(invalid_links)[:10])
else:
    print("\n✓ All linked shortage NDCs exist in drug_package.")

SHORTAGE FOREIGN KEY CHECK
Unique linked NDCs: 1393
Invalid links: 0

✓ All linked shortage NDCs exist in drug_package.


In [36]:
# ============================================================
# FINAL SHORTAGE LOAD SAFETY CHECK
# ============================================================

# Confirm MySQL shortage table is still empty
cursor.execute("SELECT COUNT(*) FROM shortage_event;")
existing_shortages = cursor.fetchone()[0]

print("FINAL SHORTAGE SAFETY CHECK")
print("=" * 50)
print("Rows ready for loading:", len(shortages_df))
print("Existing MySQL rows:", existing_shortages)
print("Invalid foreign keys:", len(invalid_links))


# SQL VARCHAR limits from our schema
varchar_limits = {
    "source_package_ndc": 20,
    "linked_package_ndc": 20,
    "generic_name": 600,
    "company_name": 255,
    "status": 100,
    "availability": 100,
    "dosage_form": 150,
    "update_type": 100
}

violations = {}

for column, limit in varchar_limits.items():

    lengths = (
        shortages_df[column]
        .dropna()
        .astype(str)
        .str.len()
    )

    too_long = (lengths > limit).sum()

    if too_long > 0:
        violations[column] = int(too_long)


print("VARCHAR violations:", violations)


# Stop automatically if anything is unsafe
assert existing_shortages == 0, \
    "STOP: shortage_event already contains data."

assert len(invalid_links) == 0, \
    "STOP: Invalid package foreign keys detected."

assert len(violations) == 0, \
    f"STOP: SQL field-size violations detected: {violations}"

assert len(shortages_df) == 1601, \
    "STOP: Unexpected shortage row count."

print("\n✓ Shortage dataset is cleared for MySQL loading.")

FINAL SHORTAGE SAFETY CHECK
Rows ready for loading: 1601
Existing MySQL rows: 0
Invalid foreign keys: 0
VARCHAR violations: {}

✓ Shortage dataset is cleared for MySQL loading.


In [37]:
# ============================================================
# LOAD SHORTAGE EVENT → MYSQL
# ============================================================

shortage_insert_sql = """
INSERT INTO shortage_event (
    source_package_ndc,
    linked_package_ndc,
    generic_name,
    company_name,
    status,
    availability,
    shortage_reason,
    therapeutic_category,
    dosage_form,
    presentation,
    update_type,
    initial_posting_date,
    update_date,
    change_date,
    resolved_note
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

shortage_rows = [
    tuple(mysql_value(value) for value in row)
    for row in shortages_df.itertuples(index=False, name=None)
]

try:
    print("Loading shortage_event...")

    cursor.executemany(
        shortage_insert_sql,
        shortage_rows
    )

    connection.commit()

    print(
        f"\n✓ SHORTAGE LOAD COMPLETE: "
        f"{len(shortage_rows):,} records inserted."
    )

except Exception as e:

    connection.rollback()

    print("\nLOAD FAILED — transaction rolled back.")
    print(type(e).__name__ + ":", e)

Loading shortage_event...

✓ SHORTAGE LOAD COMPLETE: 1,601 records inserted.


In [38]:
# ============================================================
# MYSQL SHORTAGE VALIDATION
# ============================================================

cursor.execute("""
SELECT
    COUNT(*) AS total_shortages,
    COUNT(linked_package_ndc) AS linked_rows,
    SUM(linked_package_ndc IS NULL) AS unlinked_rows,
    COUNT(DISTINCT source_package_ndc) AS unique_source_ndcs,
    COUNT(DISTINCT linked_package_ndc) AS unique_linked_ndcs
FROM shortage_event;
""")

result = cursor.fetchone()

print("MYSQL SHORTAGE VALIDATION")
print("=" * 50)
print("Total shortage events:", f"{result[0]:,}")
print("Linked shortage rows:", f"{result[1]:,}")
print("Unlinked shortage rows:", f"{result[2]:,}")
print("Unique source NDCs:", f"{result[3]:,}")
print("Unique linked NDCs:", f"{result[4]:,}")

MYSQL SHORTAGE VALIDATION
Total shortage events: 1,601
Linked shortage rows: 1,421
Unlinked shortage rows: 180
Unique source NDCs: 1,570
Unique linked NDCs: 1,393


In [39]:
# ============================================================
# BUILD FINAL RECALL EVENT DATAFRAME
# ============================================================

recalls_df = recalls[
    [
        "recall_number",
        "event_id",
        "recalling_firm",
        "classification",
        "status",
        "product_type",
        "product_description",
        "product_quantity",
        "reason_for_recall",
        "distribution_pattern",
        "voluntary_mandated",
        "initial_firm_notification",
        "recall_initiation_date",
        "report_date",
        "termination_date"
    ]
].copy()


# Text columns
recall_text_columns = [
    "recall_number",
    "recalling_firm",
    "classification",
    "status",
    "product_type",
    "product_description",
    "product_quantity",
    "reason_for_recall",
    "distribution_pattern",
    "voluntary_mandated",
    "initial_firm_notification"
]

for column in recall_text_columns:
    recalls_df[column] = recalls_df[column].apply(safe_text)


# FDA enforcement dates use YYYYMMDD format
recall_date_columns = [
    "recall_initiation_date",
    "report_date",
    "termination_date"
]

for column in recall_date_columns:
    recalls_df[column] = pd.to_datetime(
        recalls_df[column].astype("string"),
        format="%Y%m%d",
        errors="coerce"
    ).dt.date


print("RECALL EVENT READY")
print("=" * 50)
print("Rows:", len(recalls_df))
print(
    "Unique recall numbers:",
    recalls_df["recall_number"].nunique()
)
print(
    "Missing recall numbers:",
    recalls_df["recall_number"].isna().sum()
)
print(
    "Duplicate recall numbers:",
    recalls_df["recall_number"].duplicated().sum()
)

RECALL EVENT READY
Rows: 17975
Unique recall numbers: 17975
Missing recall numbers: 0
Duplicate recall numbers: 0


In [40]:
# ============================================================
# BUILD FINAL RECALL PACKAGE BRIDGE
# ============================================================

recall_packages_df = valid_recall_packages[
    [
        "recall_number",
        "package_ndc"
    ]
].copy()

recall_packages_df = recall_packages_df.rename(
    columns={"package_ndc": "source_package_ndc"}
)

# Clean identifiers
recall_packages_df["recall_number"] = (
    recall_packages_df["recall_number"]
    .astype("string")
    .str.strip()
)

recall_packages_df["source_package_ndc"] = (
    recall_packages_df["source_package_ndc"]
    .astype("string")
    .str.strip()
)

# Link against the package records actually loaded into MySQL
valid_package_ndcs = set(
    drug_packages_df["package_ndc"]
    .dropna()
    .astype(str)
)

recall_packages_df["linked_package_ndc"] = (
    recall_packages_df["source_package_ndc"]
    .where(
        recall_packages_df["source_package_ndc"].isin(
            valid_package_ndcs
        )
    )
)

print("RECALL PACKAGE BRIDGE READY")
print("=" * 50)
print("Rows:", len(recall_packages_df))
print(
    "Unique source package NDCs:",
    recall_packages_df["source_package_ndc"].nunique()
)
print(
    "Linked rows:",
    recall_packages_df["linked_package_ndc"].notna().sum()
)
print(
    "Unlinked rows:",
    recall_packages_df["linked_package_ndc"].isna().sum()
)

RECALL PACKAGE BRIDGE READY
Rows: 25577
Unique source package NDCs: 11233
Linked rows: 19673
Unlinked rows: 5904


In [41]:
# ============================================================
# FINAL RECALL INTEGRITY AUDIT
# ============================================================

valid_recall_numbers = set(
    recalls_df["recall_number"].dropna()
)

bridge_recall_numbers = set(
    recall_packages_df["recall_number"].dropna()
)

invalid_recall_links = (
    bridge_recall_numbers - valid_recall_numbers
)

duplicate_bridge_rows = (
    recall_packages_df.duplicated(
        subset=[
            "recall_number",
            "source_package_ndc"
        ]
    ).sum()
)

print("RECALL INTEGRITY AUDIT")
print("=" * 55)

print("Recall events:", len(recalls_df))
print("Recall-package relationships:", len(recall_packages_df))

print(
    "Invalid recall-number relationships:",
    len(invalid_recall_links)
)

print(
    "Duplicate recall + package relationships:",
    duplicate_bridge_rows
)

print(
    "Missing source package NDC:",
    recall_packages_df["source_package_ndc"].isna().sum()
)

print(
    "Missing recall number:",
    recall_packages_df["recall_number"].isna().sum()
)


print("\nVARCHAR SIZE CHECK")
print("-" * 55)

recall_limits = {
    "recall_number": 50,
    "recalling_firm": 255,
    "classification": 100,
    "status": 100,
    "product_type": 100,
    "voluntary_mandated": 100,
    "initial_firm_notification": 150
}

recall_violations = {}

for column, limit in recall_limits.items():

    lengths = (
        recalls_df[column]
        .dropna()
        .astype(str)
        .str.len()
    )

    too_long = int((lengths > limit).sum())

    print(
        f"{column:<30}"
        f"max={lengths.max() if len(lengths) else 0:<5} "
        f"violations={too_long}"
    )

    if too_long:
        recall_violations[column] = too_long


print("\nBridge VARCHAR check")

for column in [
    "recall_number",
    "source_package_ndc",
    "linked_package_ndc"
]:
    lengths = (
        recall_packages_df[column]
        .dropna()
        .astype(str)
        .str.len()
    )

    print(
        f"{column:<30}"
        f"max={lengths.max() if len(lengths) else 0}"
    )

RECALL INTEGRITY AUDIT
Recall events: 17975
Recall-package relationships: 25577
Invalid recall-number relationships: 0
Duplicate recall + package relationships: 0
Missing source package NDC: 0
Missing recall number: 0

VARCHAR SIZE CHECK
-------------------------------------------------------
recall_number                 max=11    violations=0
recalling_firm                max=81    violations=0
classification                max=18    violations=0
status                        max=10    violations=0
product_type                  max=5     violations=0
voluntary_mandated            max=25    violations=0
initial_firm_notification     max=81    violations=0

Bridge VARCHAR check
recall_number                 max=11
source_package_ndc            max=13
linked_package_ndc            max=12


In [42]:
# ============================================================
# BUILD FINAL RECALL EVENT DATAFRAME
# ============================================================

recalls_df = recalls[
    [
        "recall_number",
        "event_id",
        "recalling_firm",
        "classification",
        "status",
        "product_type",
        "product_description",
        "product_quantity",
        "reason_for_recall",
        "distribution_pattern",
        "voluntary_mandated",
        "initial_firm_notification",
        "recall_initiation_date",
        "report_date",
        "termination_date"
    ]
].copy()

recall_text_columns = [
    "recall_number",
    "recalling_firm",
    "classification",
    "status",
    "product_type",
    "product_description",
    "product_quantity",
    "reason_for_recall",
    "distribution_pattern",
    "voluntary_mandated",
    "initial_firm_notification"
]

for column in recall_text_columns:
    recalls_df[column] = recalls_df[column].apply(safe_text)

recall_date_columns = [
    "recall_initiation_date",
    "report_date",
    "termination_date"
]

for column in recall_date_columns:
    recalls_df[column] = pd.to_datetime(
        recalls_df[column].astype("string"),
        format="%Y%m%d",
        errors="coerce"
    ).dt.date

print("RECALL EVENT READY")
print("=" * 50)
print("Rows:", len(recalls_df))
print("Unique recall numbers:", recalls_df["recall_number"].nunique())
print("Missing recall numbers:", recalls_df["recall_number"].isna().sum())
print("Duplicate recall numbers:", recalls_df["recall_number"].duplicated().sum())

RECALL EVENT READY
Rows: 17975
Unique recall numbers: 17975
Missing recall numbers: 0
Duplicate recall numbers: 0


In [43]:
# ============================================================
# BUILD FINAL RECALL PACKAGE BRIDGE
# ============================================================

recall_packages_df = valid_recall_packages[
    [
        "recall_number",
        "package_ndc"
    ]
].copy()

# Preserve FDA's original package NDC
recall_packages_df = recall_packages_df.rename(
    columns={"package_ndc": "source_package_ndc"}
)

# Clean identifiers
recall_packages_df["recall_number"] = (
    recall_packages_df["recall_number"]
    .astype("string")
    .str.strip()
)

recall_packages_df["source_package_ndc"] = (
    recall_packages_df["source_package_ndc"]
    .astype("string")
    .str.strip()
)

# Packages that actually exist in our MySQL drug_package table
valid_package_ndcs = set(
    drug_packages_df["package_ndc"]
    .dropna()
    .astype(str)
)

# Create the foreign-key link only when a match exists
recall_packages_df["linked_package_ndc"] = (
    recall_packages_df["source_package_ndc"]
    .where(
        recall_packages_df["source_package_ndc"].isin(
            valid_package_ndcs
        )
    )
)

print("RECALL PACKAGE BRIDGE READY")
print("=" * 50)
print("Rows:", len(recall_packages_df))
print(
    "Unique source package NDCs:",
    recall_packages_df["source_package_ndc"].nunique()
)
print(
    "Linked rows:",
    recall_packages_df["linked_package_ndc"].notna().sum()
)
print(
    "Unlinked rows:",
    recall_packages_df["linked_package_ndc"].isna().sum()
)
print(
    "Unique linked package NDCs:",
    recall_packages_df["linked_package_ndc"].nunique()
)

RECALL PACKAGE BRIDGE READY
Rows: 25577
Unique source package NDCs: 11233
Linked rows: 19673
Unlinked rows: 5904
Unique linked package NDCs: 8606


In [44]:
# ============================================================
# FINAL RECALL INTEGRITY CHECK
# ============================================================

# Every bridge recall_number must exist in recall_event
valid_recall_numbers = set(
    recalls_df["recall_number"]
    .dropna()
    .astype(str)
)

bridge_recall_numbers = set(
    recall_packages_df["recall_number"]
    .dropna()
    .astype(str)
)

invalid_recall_links = (
    bridge_recall_numbers - valid_recall_numbers
)

# Check whether the exact same recall/package relationship
# appears more than once
duplicate_bridge_rows = (
    recall_packages_df.duplicated(
        subset=[
            "recall_number",
            "source_package_ndc"
        ]
    ).sum()
)

print("FINAL RECALL INTEGRITY CHECK")
print("=" * 50)

print("Recall events:", len(recalls_df))
print(
    "Recall-package relationships:",
    len(recall_packages_df)
)

print(
    "Invalid recall-number links:",
    len(invalid_recall_links)
)

print(
    "Duplicate recall/package relationships:",
    duplicate_bridge_rows
)

print(
    "Missing recall numbers:",
    recall_packages_df["recall_number"].isna().sum()
)

print(
    "Missing source package NDCs:",
    recall_packages_df["source_package_ndc"].isna().sum()
)


# Check VARCHAR fields against our MySQL schema
limits = {
    "recall_number": 50,
    "recalling_firm": 255,
    "classification": 100,
    "status": 100,
    "product_type": 100,
    "voluntary_mandated": 100,
    "initial_firm_notification": 150
}

violations = {}

print("\nVARCHAR CHECK")
print("-" * 50)

for column, limit in limits.items():

    lengths = (
        recalls_df[column]
        .dropna()
        .astype(str)
        .str.len()
    )

    max_length = lengths.max() if len(lengths) else 0
    too_long = int((lengths > limit).sum())

    print(
        f"{column:<28} "
        f"max={max_length:<5} "
        f"violations={too_long}"
    )

    if too_long > 0:
        violations[column] = too_long


print("\nSUMMARY")
print("-" * 50)
print("VARCHAR violations:", violations)

FINAL RECALL INTEGRITY CHECK
Recall events: 17975
Recall-package relationships: 25577
Invalid recall-number links: 0
Duplicate recall/package relationships: 0
Missing recall numbers: 0
Missing source package NDCs: 0

VARCHAR CHECK
--------------------------------------------------
recall_number                max=11    violations=0
recalling_firm               max=81    violations=0
classification               max=18    violations=0
status                       max=10    violations=0
product_type                 max=5     violations=0
voluntary_mandated           max=25    violations=0
initial_firm_notification    max=81    violations=0

SUMMARY
--------------------------------------------------
VARCHAR violations: {}


In [45]:
# ============================================================
# FINAL MYSQL SAFETY CHECK BEFORE RECALL LOAD
# ============================================================

cursor.execute("SELECT COUNT(*) FROM recall_event;")
existing_recall_events = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM recall_package;")
existing_recall_packages = cursor.fetchone()[0]

print("MYSQL RECALL LOAD SAFETY CHECK")
print("=" * 50)
print("Recall events ready:", f"{len(recalls_df):,}")
print("Recall-package rows ready:", f"{len(recall_packages_df):,}")
print("Existing recall_event rows:", f"{existing_recall_events:,}")
print("Existing recall_package rows:", f"{existing_recall_packages:,}")

assert existing_recall_events == 0, \
    "STOP: recall_event already contains data."

assert existing_recall_packages == 0, \
    "STOP: recall_package already contains data."

assert len(invalid_recall_links) == 0, \
    "STOP: Invalid recall relationships detected."

assert duplicate_bridge_rows == 0, \
    "STOP: Duplicate recall/package relationships detected."

assert len(violations) == 0, \
    "STOP: VARCHAR violations detected."

print("\n✓ Recall data is cleared for MySQL loading.")

MYSQL RECALL LOAD SAFETY CHECK
Recall events ready: 17,975
Recall-package rows ready: 25,577
Existing recall_event rows: 0
Existing recall_package rows: 0

✓ Recall data is cleared for MySQL loading.


In [46]:
# ============================================================
# LOAD RECALL EVENT → MYSQL
# ============================================================

recall_insert_sql = """
INSERT INTO recall_event (
    recall_number,
    event_id,
    recalling_firm,
    classification,
    status,
    product_type,
    product_description,
    product_quantity,
    reason_for_recall,
    distribution_pattern,
    voluntary_mandated,
    initial_firm_notification,
    recall_initiation_date,
    report_date,
    termination_date
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

recall_rows = [
    tuple(mysql_value(value) for value in row)
    for row in recalls_df.itertuples(index=False, name=None)
]

try:
    print("Loading recall_event...")

    cursor.executemany(
        recall_insert_sql,
        recall_rows
    )

    connection.commit()

    print(
        f"\n✓ RECALL EVENT LOAD COMPLETE: "
        f"{len(recall_rows):,} records inserted."
    )

except Exception as e:
    connection.rollback()

    print("\nLOAD FAILED — transaction rolled back.")
    print(type(e).__name__ + ":", e)

Loading recall_event...

✓ RECALL EVENT LOAD COMPLETE: 17,975 records inserted.


In [47]:
cursor.execute("""
SELECT
    COUNT(*),
    COUNT(DISTINCT recall_number)
FROM recall_event;
""")

result = cursor.fetchone()

print("RECALL EVENT MYSQL VALIDATION")
print("=" * 50)
print("Total recall events:", f"{result[0]:,}")
print("Unique recall numbers:", f"{result[1]:,}")

RECALL EVENT MYSQL VALIDATION
Total recall events: 17,975
Unique recall numbers: 17,975


In [48]:
# ============================================================
# LOAD RECALL PACKAGE → MYSQL
# ============================================================

recall_package_insert_sql = """
INSERT INTO recall_package (
    recall_number,
    source_package_ndc,
    linked_package_ndc
)
VALUES (%s, %s, %s)
"""

recall_package_rows = [
    tuple(mysql_value(value) for value in row)
    for row in recall_packages_df[
        [
            "recall_number",
            "source_package_ndc",
            "linked_package_ndc"
        ]
    ].itertuples(index=False, name=None)
]

try:
    print("Loading recall_package...")

    cursor.executemany(
        recall_package_insert_sql,
        recall_package_rows
    )

    connection.commit()

    print(
        f"\n✓ RECALL PACKAGE LOAD COMPLETE: "
        f"{len(recall_package_rows):,} relationships inserted."
    )

except Exception as e:
    connection.rollback()

    print("\nLOAD FAILED — transaction rolled back.")
    print(type(e).__name__ + ":", e)

Loading recall_package...

✓ RECALL PACKAGE LOAD COMPLETE: 25,577 relationships inserted.


In [49]:
# ============================================================
# FINAL PHARMAOPT DATABASE VALIDATION
# ============================================================

print("PHARMAOPT FDA DATABASE")
print("=" * 55)

tables = [
    "drug_product",
    "drug_package",
    "shortage_event",
    "recall_event",
    "recall_package"
]

total_records = 0

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`;")
    count = cursor.fetchone()[0]

    total_records += count

    print(f"{table:<22} {count:>10,}")

print("-" * 55)
print(f"{'TOTAL DATABASE ROWS':<22} {total_records:>10,}")


# Recall linkage validation
cursor.execute("""
SELECT
    COUNT(*) AS total,
    COUNT(linked_package_ndc) AS linked,
    SUM(linked_package_ndc IS NULL) AS unlinked
FROM recall_package;
""")

recall_validation = cursor.fetchone()

print("\nRECALL PACKAGE LINKAGE")
print("-" * 55)
print("Total relationships:", f"{recall_validation[0]:,}")
print("Linked relationships:", f"{recall_validation[1]:,}")
print("Unlinked relationships:", f"{recall_validation[2]:,}")


# Shortage linkage validation
cursor.execute("""
SELECT
    COUNT(*) AS total,
    COUNT(linked_package_ndc) AS linked,
    SUM(linked_package_ndc IS NULL) AS unlinked
FROM shortage_event;
""")

shortage_validation = cursor.fetchone()

print("\nSHORTAGE PACKAGE LINKAGE")
print("-" * 55)
print("Total shortages:", f"{shortage_validation[0]:,}")
print("Linked shortages:", f"{shortage_validation[1]:,}")
print("Unlinked shortages:", f"{shortage_validation[2]:,}")

PHARMAOPT FDA DATABASE
drug_product              138,217
drug_package              256,383
shortage_event              1,601
recall_event               17,975
recall_package             25,577
-------------------------------------------------------
TOTAL DATABASE ROWS       439,753

RECALL PACKAGE LINKAGE
-------------------------------------------------------
Total relationships: 25,577
Linked relationships: 19,673
Unlinked relationships: 5,904

SHORTAGE PACKAGE LINKAGE
-------------------------------------------------------
Total shortages: 1,601
Linked shortages: 1,421
Unlinked shortages: 180
